<a href="https://colab.research.google.com/github/MiraclePallavi/Fake_News_detection/blob/main/02model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Basic Libraries
# ==========================================
import os
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ==========================================
# Data Visualization
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# Scikit-Learn
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ==========================================
# PyTorch
# ==========================================
import torch
from torch.utils.data import Dataset

# ==========================================
# Hugging Face
# ==========================================
from datasets import Dataset as HFDataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ==========================================
# Progress Bar
# ==========================================
from tqdm.auto import tqdm

# ==========================================
# Seed for Reproducibility
# ==========================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/fakenewsProject/fakeNews_dataset.csv")

In [ ]:
len(df)

55088

In [ ]:
print(df["language"]=="russian")

0        False
1        False
2        False
3        False
4        False
         ...  
55083    False
55084    False
55085    False
55086    False
55087    False
Name: language, Length: 55088, dtype: bool


In [ ]:
duplicate_text = df["text"].duplicated().sum()

print("Duplicate texts:", duplicate_text)

Duplicate texts: 6845


In [ ]:
duplicate_df = df[df.duplicated(subset=["text"], keep=False)]

duplicate_df.sort_values("text").head(20)

,text,language,label,dataset
2521,#AnyoneButHillary: NEW POLL Shows Bernie Suppo...,english,1,ISOT
33544,#AnyoneButHillary: NEW POLL Shows Bernie Suppo...,english,1,ISOT
48684,#Austin: Fights Break Out Between Police and S...,english,1,ISOT
22781,#Austin: Fights Break Out Between Police and S...,english,1,ISOT
54719,#Berkeley CRAZY! RIOTERS CHASE And Beat People...,english,1,ISOT
53404,#Berkeley CRAZY! RIOTERS CHASE And Beat People...,english,1,ISOT
32929,#Berkeley IRONY ALERT! ANARCHISTS LOOT STARBUC...,english,1,ISOT
11734,#Berkeley IRONY ALERT! ANARCHISTS LOOT STARBUC...,english,1,ISOT
38735,#BlackLivesMatter Supporters Say No Connection...,english,1,ISOT
13855,#BlackLivesMatter Supporters Say No Connection...,english,1,ISOT


In [ ]:
df[df.duplicated(subset=["text"], keep=False)]["label"].value_counts()

,count
label,
1,10718
0,1516


In [ ]:
final_df = df.drop_duplicates()

In [ ]:
final_df.shape

(48244, 4)

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.165,
    random_state=42,
    stratify=train_df["label"]
)

In [ ]:
train_df.to_csv(
    "/content/drive/MyDrive/fakenewsProject/train.csv",
    index=False
)

val_df.to_csv(
    "/content/drive/MyDrive/fakenewsProject/val.csv",
    index=False
)

test_df.to_csv(
    "/content/drive/MyDrive/fakenewsProject/test.csv",
    index=False
)

In [ ]:
import numpy as np

token_lengths = [len(str(t).split()) for t in final_df['text']]


dataset_size = len(token_lengths)
mean_len = np.mean(token_lengths)
median_len = np.median(token_lengths)
min_len = np.min(token_lengths)
max_len = np.max(token_lengths)

p90 = np.percentile(token_lengths, 90)
p95 = np.percentile(token_lengths, 95)
p99 = np.percentile(token_lengths, 99)

# Excess counts (using 512 as the word-count heuristic threshold)
count_over_512 = sum(1 for length in token_lengths if length > 512)
percentage_over_512 = (count_over_512 / dataset_size) * 100

# Report
print(f"--- Estimated Length Statistics (based on word count) ---")
print(f"Dataset size: {dataset_size}")
print(f"Mean length: {mean_len:.2f}")
print(f"Median length: {median_len:.2f}")
print(f"Minimum length: {min_len}")
print(f"Maximum length: {max_len}")
print(f"90th percentile: {p90:.2f}")
print(f"95th percentile: {p95:.2f}")
print(f"99th percentile: {p99:.2f}")
print(f"Number of samples > 512: {count_over_512}")
print(f"Percentage of samples > 512: {percentage_over_512:.2f}%")

--- Estimated Length Statistics (based on word count) ---
Dataset size: 48244
Mean length: 454.20
Median length: 381.00
Minimum length: 2
Maximum length: 8148
90th percentile: 823.00
95th percentile: 1047.00
99th percentile: 2463.71
Number of samples > 512: 13930
Percentage of samples > 512: 28.87%


In [ ]:
from transformers import DataCollatorWithPadding

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
def tokenize_function(data):
    return tokenizer(
        data["text"],
        truncation=True,
        max_length=512,
        padding=False
    )

In [ ]:
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/36798 [00:00<?, ? examples/s]

Map:   0%|          | 0/7272 [00:00<?, ? examples/s]

Map:   0%|          | 0/11018 [00:00<?, ? examples/s]

In [ ]:
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

In [ ]:
train_dataset = train_dataset.remove_columns(
    ["text", "language", "dataset"]
)

val_dataset = val_dataset.remove_columns(
    ["text", "language", "dataset"]
)

test_dataset = test_dataset.remove_columns(
    ["text", "language", "dataset"]
)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
print(model.config)

XLMRobertaConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "xlm-roberta-base",
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.49.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 250002
}



In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=2e-5,

    weight_decay=0.01,

    warmup_ratio=0.1,

    fp16=True,

    logging_strategy="steps",
    logging_steps=100,

    evaluation_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    save_total_limit=2,

    report_to="none"
)

In [ ]:
train_dataset = train_dataset.remove_columns("__index_level_0__")
val_dataset = val_dataset.remove_columns("__index_level_0__")
test_dataset = test_dataset.remove_columns("__index_level_0__")

In [ ]:
print(train_dataset[0].keys())

dict_keys(['labels', 'input_ids', 'attention_mask'])


In [ ]:
batch = data_collator([
    train_dataset[540],
    train_dataset[539]
])

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([2, 432])
torch.Size([2, 432])
torch.Size([2])


In [ ]:
from transformers import Trainer
trainer =Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    tokenizer = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.020600,0.020200,0.995462,0.995478,0.995462,0.995462
2,0.009400,0.021405,0.996287,0.996306,0.996287,0.996287
3,0.001300,0.012126,0.997525,0.997525,0.997525,0.997525


TrainOutput(global_step=6900, training_loss=0.03840401750435864, metrics={'train_runtime': 4038.5579, 'train_samples_per_second': 27.335, 'train_steps_per_second': 1.709, 'total_flos': 2.904082929092688e+16, 'train_loss': 0.03840401750435864, 'epoch': 3.0})

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/FakeNewsModel"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

('/content/drive/MyDrive/FakeNewsModel/tokenizer_config.json',
 '/content/drive/MyDrive/FakeNewsModel/special_tokens_map.json',
 '/content/drive/MyDrive/FakeNewsModel/sentencepiece.bpe.model',
 '/content/drive/MyDrive/FakeNewsModel/added_tokens.json',
 '/content/drive/MyDrive/FakeNewsModel/tokenizer.json')